# 月データ探索ツール（標準編）

公開データセットを、**自分でX軸・Y軸を選びながら**散布図で見比べる教材です。
情報Ⅰの範囲（散布図・基本統計量・相関係数）を超える内容（検定・回帰・疑似相関の検証など）は
`explore_advanced.ipynb`（発展編）にまとめてあります。

使い方：
1. 上から順にセルを実行する（Colabなら「ランタイム」→「すべてのセルを実行」）
2. 一番下に出てくるプルダウンで、データセット・X軸・Y軸・色分けを自由に選ぶ
3. まずは何も予想せず、いろいろな組み合わせを試してみる
4. 「気になる関係」が見つかったら、ワークシート（docs/worksheet.pdf）に書き出す

> 迷ったら、いちばん下の「問いのヒント」を参考にしてください。

In [7]:
# ライブラリの読み込みと実行環境の確認
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import ipywidgets as widgets
from IPython.display import display, clear_output

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB and not os.path.exists('data'):
    print("⚠️ dataフォルダが見つかりません。")
    print("Colabでこのノートブックだけを開いた場合、データファイルは一緒に来ません。")
    print("次の2行のコメントを外して実行し、リポジトリごと取得してください：")
    print("  # !git clone https://github.com/<ユーザー名>/<リポジトリ名>.git")
    print("  # %cd <リポジトリ名>/notebooks")

# 日本語フォントの設定
# matplotlibの標準フォントには日本語が含まれておらず、グラフの日本語ラベルが
# 文字化け（豆腐□）することがある。Windows/Colabどちらでも確実に表示できるよう、
# フォントファイル（Noto Sans JP）をこのリポジトリに同梱して読み込む。
_font_path = None
for _p in ('assets/NotoSansJP-Regular.ttf', 'notebooks/assets/NotoSansJP-Regular.ttf'):
    if os.path.exists(_p):
        _font_path = _p
        break
if _font_path:
    fm.fontManager.addfont(_font_path)
    plt.rcParams['font.family'] = fm.FontProperties(fname=_font_path).get_name()
else:
    print("⚠️ 日本語フォント(assets/NotoSansJP-Regular.ttf)が見つかりません。"
          "グラフの日本語表示が文字化けする可能性があります。")

In [8]:
def _read(name):
    for p in (f'../data/{name}', f'data/{name}'):
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(name)

craters = _read('craters_subset.csv')
craters_3d = _read('craters_3d.csv')
deepcraters = _read('deepcraters.csv')
diviner = _read('diviner_global.csv')
maria = _read('maria_boundaries.csv')
ephemeris = _read('moon_ephemeris.csv')
polar_illum = _read('lola_polar_illumination.csv')
geology = _read('moon_geology_grid.csv')

# DeepCratersの年代インデックス（1〜5の数字）を、意味のわかる文字列にした列を追加しておく
AGE_NAMES = {
    1: '1:Pre-Nectarian(最も古い)',
    2: '2:Nectarian',
    3: '3:Imbrian',
    4: '4:Eratosthenian',
    5: '5:Copernican(最も新しい)',
}
deepcraters['Age_name'] = deepcraters['Age'].map(AGE_NAMES)

# moon_ephemeris.csv（日付の文字列）は散布図のX軸に使えないので、
# 「観測開始日から何日目か」という通し番号の列を追加しておく
ephemeris['day_index'] = range(len(ephemeris))

# データセットごとに「どの列を選べるか」「日本語での説明」を定義する
# ※ Robbins Crater DBには「深さ」の情報は含まれていません（要確認事項として実データを確認した結果、
#    緯度・経度・直径に関する列のみで、深さを表す列は存在しませんでした）。深さを含む別カタログ
#    （Wang & Wu, 2021）を craters_3d として別データセットに用意しています。
datasets = {
    'クレーターの直径・形（Robbins Crater DB, 直径8km以上）': {
        'df': craters,
        'columns': {
            'lat': '緯度 [度]',
            'lon': '経度 [度]',
            'diam_km': '直径 [km]',
            'diam_major_km': '長径 [km]',
            'diam_minor_km': '短径 [km]',
            'eccentricity': '離心率（真円=0に近いほど丸い）',
            'ellipticity': '扁平率（真円=1に近いほど丸い）',
            'rim_arc_fraction': 'リムが検出できた割合（0〜1）',
        },
        'color_options': [],
        'latlon': ('lon', 'lat'),
    },
    'クレーターの直径と深さ（Wang & Wu 2021, 直径10km以上）': {
        'df': craters_3d,
        'columns': {
            'lat': '緯度 [度]',
            'lon': '経度 [度]',
            'diameter_km': '直径 [km]',
            'depth_km': '深さ [km]',
            'depth_diameter_ratio': '深さ÷直径の比',
        },
        'color_options': [],
        'latlon': ('lon', 'lat'),
    },
    'クレーターの推定年代（DeepCraters, 直径8km以上）': {
        'df': deepcraters,
        'columns': {
            'Lat': '緯度 [度]',
            'Lon': '経度 [度]',
            'Diam_km': '直径 [km]',
            'Age': '推定年代区分（1〜5、数字が大きいほど新しい）',
        },
        'color_options': ['Age_name', 'Flags_data'],
        'latlon': ('Lon', 'Lat'),
    },
    '月面の温度（Diviner, 全球0.5度グリッド）': {
        'df': diviner,
        'columns': {
            'lon': '経度 [度]',
            'lat': '緯度 [度]',
            'temp_noon_K': '正午の温度 [K]',
            'temp_midnight_K': '深夜0時の温度 [K]',
            'temp_diff_K': '昼夜の温度差 [K]',
        },
        'color_options': [],
        'latlon': ('lon', 'lat'),
    },
    '月の南極・北極の日照（LOLA, 約1kmグリッド）': {
        'df': polar_illum,
        'columns': {
            'lon': '経度 [度]',
            'lat': '緯度 [度]（正=北極側、負=南極側）',
            'average_illumination_percent': '平均日照率 [%]（高いほど太陽光発電向き）',
            'permanent_shadow_fraction': '永久影である割合（0〜1、高いほど氷が残りやすい）',
        },
        'color_options': [],
        'latlon': ('lon', 'lat'),
    },
    '月の地質年代・地形マップ（USGS統合地質図, 1度グリッド）': {
        'df': geology,
        'columns': {
            'lon_grid': '経度 [度]',
            'lat_grid': '緯度 [度]',
        },
        'color_options': ['relative_age', 'terrain_type'],
        'latlon': ('lon_grid', 'lat_grid'),
    },
    '月の海・大洋の分布（USGS地名辞典）': {
        'df': maria,
        'columns': {
            'center_lon': '中心経度 [度]',
            'center_lat': '中心緯度 [度]',
            'radius_km': '概算半径 [km]',
        },
        'color_options': [],
        'latlon': ('center_lon', 'center_lat'),
    },
    '地球から見た月（天体暦, 過去5年・日次）': {
        'df': ephemeris,
        'columns': {
            'day_index': '観測開始日からの経過日数',
            'distance_km': '地球ー月の距離 [km]',
            'apparent_diameter_arcsec': '見かけの直径 [秒角]',
            'phase_angle': '位相角 [度]（0=満月に近い, 180=新月に近い）',
            'illumination_fraction': '輝面比（0=新月, 1=満月）',
        },
        'color_options': [],
        'latlon': None,
    },
}

print('読み込み完了：')
for name, d in datasets.items():
    print(f' - {name}: {len(d["df"]):,} 件')

読み込み完了：
 - クレーターの直径・形（Robbins Crater DB, 直径8km以上）: 36,377 件
 - クレーターの直径と深さ（Wang & Wu 2021, 直径10km以上）: 24,982 件
 - クレーターの推定年代（DeepCraters, 直径8km以上）: 18,996 件
 - 月面の温度（Diviner, 全球0.5度グリッド）: 259,200 件
 - 月の南極・北極の日照（LOLA, 約1kmグリッド）: 157,922 件
 - 月の地質年代・地形マップ（USGS統合地質図, 1度グリッド）: 64,800 件
 - 月の海・大洋の分布（USGS地名辞典）: 23 件
 - 地球から見た月（天体暦, 過去5年・日次）: 1,827 件


## 探索ツール

下のプルダウンでデータセットとX軸・Y軸を選ぶと、その場で散布図と基本統計量（平均・標準偏差・相関係数）が表示されます。
点の数が多いデータセットは、見やすさのため一部だけをランダムに抜き出して表示します（「表示点数の上限」で調整可）。

緯度・経度を軸に選んだときは、「月の海の位置を重ねる」にチェックを入れると、
主要な海・大洋の中心位置（USGS地名辞典より）が赤い&times;印で重なって表示されます。

In [9]:
dataset_dropdown = widgets.Dropdown(
    options=list(datasets.keys()),
    description='データセット:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='560px'),
)
x_dropdown = widgets.Dropdown(description='X軸:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'))
y_dropdown = widgets.Dropdown(description='Y軸:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'))
color_dropdown = widgets.Dropdown(description='色分け:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'))
sample_slider = widgets.IntSlider(
    value=3000, min=500, max=20000, step=500,
    description='表示点数の上限:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='560px'),
)
overlay_maria = widgets.Checkbox(value=False, description='月の海の位置を重ねる（緯度経度のときのみ有効）', indent=False)
logx_check = widgets.Checkbox(value=False, description='X軸を対数目盛りにする', indent=False)
logy_check = widgets.Checkbox(value=False, description='Y軸を対数目盛りにする', indent=False)
output = widgets.Output()


def update_variable_options(change=None):
    info = datasets[dataset_dropdown.value]
    options = [(label, col) for col, label in info['columns'].items()]
    x_dropdown.options = options
    y_dropdown.options = options
    x_dropdown.value = options[0][1]
    y_dropdown.value = options[1][1] if len(options) > 1 else options[0][1]
    color_dropdown.options = [('なし', None)] + [(c, c) for c in info['color_options']]
    color_dropdown.value = None
    draw_plot()


def draw_plot(change=None):
    with output:
        clear_output(wait=True)
        info = datasets[dataset_dropdown.value]
        df = info['df']
        x_col, y_col = x_dropdown.value, y_dropdown.value
        color_col = color_dropdown.value

        n = min(len(df), sample_slider.value)
        plot_df = df.sample(n=n, random_state=0) if len(df) > n else df

        fig, ax = plt.subplots(figsize=(7, 6))
        if color_col:
            categories = plot_df[color_col].astype('category')
            sc = ax.scatter(plot_df[x_col], plot_df[y_col], c=categories.cat.codes,
                             cmap='viridis', s=8, alpha=0.6)
            handles, _ = sc.legend_elements()
            ax.legend(handles, categories.cat.categories, title=color_col,
                       bbox_to_anchor=(1.05, 1), loc='upper left')
        else:
            ax.scatter(plot_df[x_col], plot_df[y_col], s=8, alpha=0.4)

        # 緯度経度の軸を選んでいて、かつチェックがオンなら、月の海の位置を重ねる
        latlon = info.get('latlon')
        if overlay_maria.value and latlon and x_col == latlon[0] and y_col == latlon[1]:
            ax.scatter(maria['center_lon'], maria['center_lat'], marker='x', c='crimson',
                       s=60, linewidths=1.5, label='月の海・大洋（中心位置）')
            ax.legend(loc='upper right', fontsize=8)

        try:
            if logx_check.value:
                ax.set_xscale('log')
            if logy_check.value:
                ax.set_yscale('log')
        except Exception:
            print('\u26a0\ufe0f このデータには0以下の値が含まれているため、対数グラフに変換できません。')

        ax.set_xlabel(info['columns'].get(x_col, x_col))
        ax.set_ylabel(info['columns'].get(y_col, y_col))
        ax.set_title(f"{dataset_dropdown.value}\n(表示 {n:,} / 全 {len(df):,} 件)")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

        if x_col != y_col:
            corr = plot_df[[x_col, y_col]].corr().iloc[0, 1]
            print(f'相関係数 r = {corr:.3f}')
        stats = plot_df[[x_col, y_col]].describe().loc[['mean', 'std', 'min', 'max']]
        display(stats)


dataset_dropdown.observe(update_variable_options, names='value')
x_dropdown.observe(draw_plot, names='value')
y_dropdown.observe(draw_plot, names='value')
color_dropdown.observe(draw_plot, names='value')
sample_slider.observe(draw_plot, names='value')
overlay_maria.observe(draw_plot, names='value')
logx_check.observe(draw_plot, names='value')
logy_check.observe(draw_plot, names='value')

update_variable_options()

display(widgets.VBox([
    dataset_dropdown,
    widgets.HBox([x_dropdown, y_dropdown, color_dropdown]),
    sample_slider,
    widgets.HBox([overlay_maria, logx_check, logy_check]),
    output,
]))

## 問いのヒント（迷ったときに）

正解ではなく、あくまで出発点の例です。自分で見つけた組み合わせを優先してください。

1. クレーターの直径と深さの関係（「クレーターの直径と深さ（Wang & Wu 2021）」データセットを使う）
2. クレーターの直径と、離心率・扁平率（＝どれくらい丸いか）の関係
3. クレーターの緯度・経度分布のかたより（「月の海の位置を重ねる」を使うと、海の上でクレーターが少ない場所があるか確認しやすい）
4. 緯度と正午の温度の関係
5. 同じ場所での「正午の温度」と「深夜0時の温度」の差（昼夜温度差）と、緯度との関係
6. クレーターの推定年代（Age）と、直径や分布との関係（DeepCratersのデータのみで完結）
7. 地球ー月の距離と、見かけの直径の関係（近いほど大きく見える？）
8. 経過日数と距離の関係（周期的な変化が見えるか）
9. 月の南極・北極で、太陽光発電に向いた場所を探す（日照率が高く、永久影の割合が低い場所はどこか）
10. 地質年代マップで「色分け：relative_age」を選び、どの時代の地形が多く残っているか確認する
11. 地質年代マップで「色分け：terrain_type」を選び、Highland（高地）とMare（海）の分布を確認する

> **注記**：当初の教材案が参照していたRobbins Crater Database (2018) には、実際に確認したところ
> **深さ（Depth）を表す列は含まれていません**（緯度・経度・直径・形状に関する列のみ）。
> そのため「直径と深さ」を調べたい場合は、深さを含む別カタログ（Wang & Wu, 2021）の
> データセットを使ってください。
>
> **月齢と地球環境（地震・潮汐力）の関係を調べたい人へ**：`explore_advanced.ipynb`（発展編）に
> 専用のセクションがあります。ただし相関係数の解釈には注意が必要な内容なので、発展編の説明を
> よく読んでから取り組んでください。

## データの出典

- Robbins, S. J. (2018). *A New Global Database of Lunar Impact Craters >1–2 km*. USGS Astrogeology Science Center.
  https://astrogeology.usgs.gov/search/map/Moon/Research/Craters/lunar_crater_database_robbins_2018
- Wang, Y., Wu, B. (2021). *An improved global catalog of lunar impact craters (≥1 km) with 3D
  morphometric information*. JGR Planets, 126, e2020JE006728. Zenodo:
  https://doi.org/10.5281/zenodo.4983248 （クレーターの深さを含むカタログ、CC BY 4.0）
- Yang, C., Guan, R. (2020). *CE_DeepCraters* (Aged Lunar Crater Database). figshare.
  https://doi.org/10.6084/m9.figshare.12768539
- Williams, J.-P. et al. (2017). *The global surface temperatures of the Moon as measured by the
  Diviner Lunar Radiometer Experiment*. Icarus, 283, 300-325. データ配布：
  https://www.diviner.ucla.edu/data （UCLA Diviner Lunar Radiometer Experiment チーム提供、
  0.5 ppd全球ラスタープロダクト）
- Mazarico, E. et al. (2011). *Illumination conditions of the lunar polar regions using LOLA
  topography*. Icarus, 211, 1066-1081. データ配布：LRO LOLA Team (NASA GSFC), PDS Geosciences
  Node（南極・北極の平均日照率・永久影マップ、約1kmグリッドに再集計）
- Fortezzo, C. M. et al. (2020). *Release of the Digital Unified Global Geologic Map of the Moon
  at 1:5,000,000-scale*. USGS Astrogeology Science Center.
  https://astrogeology.usgs.gov/search/map/Moon/Geology/Unified_Geologic_Map_of_the_Moon_GIS_v2
  （地質年代・地形区分。GISポリゴンを1度グリッドに再集計）
- USGS Astrogeology Science Center. *Gazetteer of Planetary Nomenclature: Moon*.
  https://planetarynames.wr.usgs.gov/ （月の海・大洋の中心座標・直径）
- NASA JPL Solar System Dynamics. *HORIZONS System* （地球ー月の距離・見かけの直径・位相角・輝面比）。
  https://ssd.jpl.nasa.gov/horizons/